# FinBERT Failure-Mode Analysis (Paper / TDS)

**Purpose:** Publishable analysis of the five failure-mode fixes (negation, conditionals, hedged language, numeric comparison, ABSA). Produces: (1) **ablation F1 table** (baseline + each fix in isolation + all fixes) across full and per-subset test set, (2) **per-category error analysis** with 5 examples each (text, baseline wrong → fixed correct), (3) **negation coverage** — fraction of negation subset covered by the five regex patterns vs spaCy path, (4) **inter-category overlap** cross-tabulation. No retraining; loads model from 04a and test split from 03.

**Prerequisites:** Run `03_sentiment_FP_FiQA_feature_engineering.ipynb` (test.parquet) and `04a_sentiment_finbert_training.ipynb` (saves `models/sentiment/finbert_tuned_v1/`). Optional: `data/proof/credit_risk_sentiment/eval_failure_modes.json` from 04a Section 6 for sanity check; this notebook reruns all seven configurations for the ablation table.

## 1. Setup and load test data (from 03) and model path (from 04a)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "credit_risk").is_dir() and (ROOT / "data").is_dir():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

processed_dir = ROOT / "data" / "credit_risk_sentiment" / "processed"
model_dir = ROOT / "models" / "sentiment" / "finbert_tuned_v1"
if not (processed_dir / "test.parquet").exists():
    raise FileNotFoundError("Run 03_sentiment_FP_FiQA_feature_engineering.ipynb first.")

test_df = pd.read_parquet(processed_dir / "test.parquet")
texts = test_df["text"].astype(str).tolist()
y_true = test_df["label"].astype(str).str.strip().str.lower().tolist()
model_path = str(model_dir) if model_dir.exists() else None
print("Test n =", len(texts), "| Model path:", model_path or "ProsusAI/finbert (pretrained)")

## 2. Subset masks and seven pipeline configurations

Define subset detectors (same as 03/04a). Run seven configs: **Baseline** (all off), **+Negation only**, **+Conditional only**, **+Numeric only**, **+Hedge only**, **+ABSA only**, **All fixes**. Collect predictions and run `evaluate_on_subsets` for each.

In [ ]:
from credit_risk.sentiment import SentimentPipeline, SentimentPipelineConfig
from credit_risk.sentiment.evaluation import evaluate_on_subsets
from credit_risk.sentiment.negation import NegationHandler
from credit_risk.sentiment.conditional import ConditionalDetector
from credit_risk.sentiment.numeric_comparison import NumericComparisonExtractor
from credit_risk.sentiment.hedging import HedgeScorer

neg_handler = NegationHandler()
cond_detector = ConditionalDetector()
num_extractor = NumericComparisonExtractor(underperform_pct=5.0)
hedge_scorer = HedgeScorer()

negation_fn = lambda t: neg_handler.has_negation(t)
conditional_fn = lambda t: cond_detector.is_conditional(t)
numeric_fn = lambda t: num_extractor.extract(t) is not None
hedged_fn = lambda t: hedge_scorer.has_hedge(t)

def make_config(neg=False, cond=False, num=False, hedge=False, absa=False):
    c = SentimentPipelineConfig(
        use_negation_handling=neg,
        use_conditional_detection=cond,
        use_numeric_comparison=num,
        use_hedge_adjustment=hedge,
        use_entity_sentiment=absa,
    )
    c.model_path = model_path
    return c

configs = {
    "Baseline": make_config(False, False, False, False, False),
    "+Negation only": make_config(True, False, False, False, False),
    "+Conditional only": make_config(False, True, False, False, False),
    "+Numeric only": make_config(False, False, True, False, False),
    "+Hedge only": make_config(False, False, False, True, False),
    "+ABSA only": make_config(False, False, False, False, True),
    "All fixes": make_config(True, True, True, True, True),
}

predictions = {}
for name, cfg in configs.items():
    pipe = SentimentPipeline(config=cfg)
    preds = [pipe.predict(t)["sentence_sentiment"] for t in texts]
    predictions[name] = preds

results_per_config = {}
for name, preds in predictions.items():
    results_per_config[name] = evaluate_on_subsets(
        texts, y_true, preds,
        negation_fn=negation_fn,
        conditional_fn=conditional_fn,
        numeric_fn=numeric_fn,
        hedged_fn=hedged_fn,
    )
print("Seven configurations run; results_per_config and predictions ready.")

## 3. Ablation F1 table (publishable)

One table: rows = configuration, columns = full + negation + conditional + numeric_comparison + hedged. Each cell = F1 macro (n) for that subset.

In [ ]:
subsets = ["full", "negation", "conditional", "numeric_comparison", "hedged"]
rows = []
for config_name in configs.keys():
    r = results_per_config[config_name]
    row = {"Configuration": config_name}
    for sub in subsets:
        if sub in r:
            row[sub] = f"{r[sub]['f1']:.3f} (n={r[sub]['n']})"
        else:
            row[sub] = "—"
    rows.append(row)
ablation_df = pd.DataFrame(rows)
ablation_df

In [ ]:
# Numeric-only table (for paper/TDS: copy-paste or export)
ablation_numeric = pd.DataFrame({
    config_name: {sub: results_per_config[config_name].get(sub, {}).get("f1", np.nan) for sub in subsets}
    for config_name in configs.keys()
}).T
ablation_numeric.index.name = "Configuration"
ablation_numeric.round(3)

## 4. Per-category error analysis (5 examples each)

For each failure-mode subset: show up to 5 examples where **baseline** predicted wrong and **All fixes** predicted correct. Display: text, ground truth, baseline pred, fixed pred.

In [ ]:
baseline_pred = predictions["Baseline"]
fixed_pred = predictions["All fixes"]
y_true_arr = np.array(y_true)

def get_subset_mask(fn):
    return np.array([bool(fn(t)) for t in texts])

subset_masks = {
    "negation": get_subset_mask(negation_fn),
    "conditional": get_subset_mask(conditional_fn),
    "numeric_comparison": get_subset_mask(numeric_fn),
    "hedged": get_subset_mask(hedged_fn),
}

def normalize(s):
    return (s or "").strip().lower()

error_examples = {}
for cat, mask in subset_masks.items():
    wrong_right = []  # baseline wrong, fixed correct
    for i in np.where(mask)[0]:
        if normalize(baseline_pred[i]) != normalize(y_true_arr[i]) and normalize(fixed_pred[i]) == normalize(y_true_arr[i]):
            wrong_right.append({
                "text": texts[i][:200] + ("..." if len(texts[i]) > 200 else ""),
                "ground_truth": y_true_arr[i],
                "baseline_pred": baseline_pred[i],
                "fixed_pred": fixed_pred[i],
            })
        if len(wrong_right) >= 5:
            break
    error_examples[cat] = wrong_right

In [ ]:
for cat, examples in error_examples.items():
    print("=" * 60)
    print(f"Category: {cat} (baseline wrong → fixed correct)")
    print("=" * 60)
    for j, ex in enumerate(examples, 1):
        print(f"\n  Example {j}:")
        print(f"    Text: {ex['text']}")
        print(f"    GT: {ex['ground_truth']}  |  Baseline: {ex['baseline_pred']}  →  Fixed: {ex['fixed_pred']}")
    if not examples:
        print("  (No examples in this subset where baseline wrong and fixed correct.)")
    print()

## 5. Negation coverage analysis (regex vs spaCy path)

The negation fix uses (a) five **regex patterns** (`NEGATION_FLIP_PHRASES`) for preprocessing and (b) **spaCy** for detection and optional label flip. Report: of the test sentences in the negation subset, how many are covered by at least one of the five regex patterns? How many are detected by spaCy? This documents a limitation: if regex covers only a fraction of negation cases, the rest rely on spaCy or label flip.

In [ ]:
from credit_risk.sentiment.negation import NEGATION_FLIP_PHRASES

def covered_by_regex_phrases(text):
    """True if text matches at least one of the five NEGATION_FLIP_PHRASES patterns (preprocessing path)."""
    return any(p.search(text) for p, _ in NEGATION_FLIP_PHRASES)

neg_mask = get_subset_mask(negation_fn)
neg_indices = np.where(neg_mask)[0]
n_neg = len(neg_indices)

regex_covered = sum(1 for i in neg_indices if covered_by_regex_phrases(texts[i]))
not_regex_covered = n_neg - regex_covered

print("Negation subset (test set): n =", n_neg)
print("  Covered by at least one of 5 regex patterns (NEGATION_FLIP_PHRASES):", regex_covered, f"({100*regex_covered/max(1,n_neg):.1f}%)")
print("  Not covered by regex (rely on spaCy detection + label flip):", not_regex_covered)
print("  SpaCy available:", neg_handler.available)
print("\nLimitation: Only 5 regex patterns; remaining negation cases rely on spaCy detection + post-hoc label flip. If spaCy is unavailable, only the regex path applies.")

## 6. Inter-category overlap cross-tabulation

How many test samples trigger multiple failure modes? Sentences with both negation and hedging (or conditional + numeric) are harder than either alone. Build a cross-tabulation of counts.

In [ ]:
flags_df = pd.DataFrame({
    "negation": [negation_fn(t) for t in texts],
    "conditional": [conditional_fn(t) for t in texts],
    "numeric_comparison": [numeric_fn(t) for t in texts],
    "hedged": [hedged_fn(t) for t in texts],
})
flags_df["n_flags"] = flags_df.sum(axis=1)

print("Count by number of failure-mode flags (0–4) per sentence:")
print(flags_df["n_flags"].value_counts().sort_index().to_string())
print("\nPairwise overlap counts (sentences with both):")
from itertools import combinations
for a, b in combinations(["negation", "conditional", "numeric_comparison", "hedged"], 2):
    n_both = (flags_df[a] & flags_df[b]).sum()
    print(f"  {a} & {b}: {n_both}")

## 7. Optional: sanity check vs eval_failure_modes.json

If 04a Section 6 was run, I can compare this notebook’s Baseline and All-fixes F1 to the saved JSON (same configs, so numbers should match).

In [ ]:
import json
eval_path = ROOT / "data" / "proof" / "credit_risk_sentiment" / "eval_failure_modes.json"
if eval_path.exists():
    with open(eval_path) as f:
        saved = json.load(f)
    print("Saved baseline (from 04a):", saved.get("baseline", {}))
    print("Saved with_fixes (from 04a):", saved.get("with_fixes", {}))
    print("\nThis notebook Baseline full F1:", results_per_config["Baseline"].get("full", {}).get("f1"))
    print("This notebook All fixes full F1:", results_per_config["All fixes"].get("full", {}).get("f1"))
else:
    print("eval_failure_modes.json not found; run 04a Section 6 to generate it.")